# Predicting Mental Health Treatment-Seeking Among Technology Workers Using Machine Learning

## Part 1 Introduction and Research Objectives

### 1.1 Dataset introduction

This study uses the mental health  in Tech Survey in 2014 from OSMI (Open Sourcing Mental illness), obtained from Kaggle.

This survey mainly explores tech worker's experiences with mental health, their working conditions, and their attitudes toward mental health issues.

The original dataset has 1,259 records and 27 variables, including information such as age, gender, family history of mental illness and working environment.

Dataset source: [https://www.kaggle.com/datasets/osmi/mental-health-in-tech-survey](https://www.kaggle.com/datasets/osmi/mental-health-in-tech-survey)

Original survey organisation: [https://www.osmihelp.org/research.html](https://www.osmihelp.org/research.html)


### 1.2 Research Problem and Objectives

 #### 1.2.1 Research Motivation

Mental health Treatment-seeking in technology-related workplaces is studying because personal and workplaces factors may be linked to whether people seek professional help.

Uderstanding which factors are assciated with treatment seeking may prvide useful information about mental health support needs in technology related workplaces.

By analysing this data, I want to examine how personal and workplace-related factors are associated with responsdents' reported mental health treatment seeking. I also compare how well different machine learning models perform on this classification task.

#### 1.2.2 Research Questions

Main question: Using survey information about people's personal background and their work environment, how well can machine learning models classify whether tech workers have sought mental health treatment?

SQ1: Which survey features are statistical association with whether respondents have sought mental health treatment? And which features are most important for classification?

SQ2: How does the performance of Logistic Regression and Random Forest change with different train-test splits and hyperparameter tuning? Which model perform better, Logistic Regression or Random Forest, and whiy? Can these model also perform well on unseen data?

#### 1.2.3 Target Variable and Task Type

This study uses treatment as the target variable. It shows whether respondents have reported seeking treatment for a mental health confition, with two possible categories: Yes or No.

Therefore, this is a superised binary claasification task rather than a regression task, because the target variable is categorical rather than continuous.

#### 1.2.4 Research Objectives
Three main objectives.
1.Explore the OSMI survey dataset, check data quality, and carry out necessary cleaning and preprocessing.

2.Build and compare two classification models, Logistic Regression and Random Forest. I will test three train-test split ratios (5%,15%,25%) and use cross-validation to assess model performance.

3.Evaluate different hyperparameter setting to find suitable configurations for the models. I will also examine statistical associations between survey featuresa and treatment-seeking, analyse feature importance, and evaluate the models' generalisation ability. 

### 1.3 Dataset Loading and Initial Inspection

In [1]:
#Importing necessary Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 

C:\Users\Lenovo\anaconda3\lib\site-packages\scipy\__init__.py:138: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.3)
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion} is required for this version of "


In [2]:
#Loading Dataset
df = pd.read_csv("mental-heath-in-tech-2014.csv")

In [3]:
# Display first 5 records
df.head()

,Timestamp,Age,Gender,Country,state,self_employed,family_history,treatment,work_interfere,no_employees,...,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_health_interview,phys_health_interview,mental_vs_physical,obs_consequence,comments
0,2014-08-27 11:29:31,37,Female,United States,IL,NaN,No,Yes,Often,6-25,...,Somewhat easy,No,No,Some of them,Yes,No,Maybe,Yes,No,NaN
1,2014-08-27 11:29:37,44,M,United States,IN,NaN,No,No,Rarely,More than 1000,...,Don't know,Maybe,No,No,No,No,No,Don't know,No,NaN
2,2014-08-27 11:29:44,32,Male,Canada,NaN,NaN,No,No,Rarely,6-25,...,Somewhat difficult,No,No,Yes,Yes,Yes,Yes,No,No,NaN
3,2014-08-27 11:29:46,31,Male,United Kingdom,NaN,NaN,Yes,Yes,Often,26-100,...,Somewhat difficult,Yes,Yes,Some of them,No,Maybe,Maybe,No,Yes,NaN
4,2014-08-27 11:30:22,31,Male,United States,TX,NaN,No,No,Never,100-500,...,Don't know,No,No,Some of them,Yes,Yes,Yes,Don't know,No,NaN


In [4]:
# Display the last 5 records
df.tail()

,Timestamp,Age,Gender,Country,state,self_employed,family_history,treatment,work_interfere,no_employees,...,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_health_interview,phys_health_interview,mental_vs_physical,obs_consequence,comments
1254,2015-09-12 11:17:21,26,male,United Kingdom,NaN,No,No,Yes,NaN,26-100,...,Somewhat easy,No,No,Some of them,Some of them,No,No,Don't know,No,NaN
1255,2015-09-26 01:07:35,32,Male,United States,IL,No,Yes,Yes,Often,26-100,...,Somewhat difficult,No,No,Some of them,Yes,No,No,Yes,No,NaN
1256,2015-11-07 12:36:58,34,male,United States,CA,No,Yes,Yes,Sometimes,More than 1000,...,Somewhat difficult,Yes,Yes,No,No,No,No,No,No,NaN
1257,2015-11-30 21:25:06,46,f,United States,NC,No,No,No,NaN,100-500,...,Don't know,Yes,No,No,No,No,No,No,No,NaN
1258,2016-02-01 23:04:31,25,Male,United States,IL,No,Yes,Yes,Sometimes,26-100,...,Don't know,Maybe,No,Some of them,No,No,No,Don't know,No,NaN


From df.head()and df.tail(), I noticed some data quality issues. The Gender column has different entries, such as M,Male, male, and f. 
Some missing values(NaN) also appear in state, self_employed, and work_interfere.

In [5]:
df.shape

(1259, 27)

df.shape shows the raw data has 1,259 rows and 27 columns.

The data contains both numeric and categorical features like Age and Gender. Some values are missing , and Gender entries are inconsistent.

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1259 entries, 0 to 1258
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Timestamp                  1259 non-null   object
 1   Age                        1259 non-null   int64 
 2   Gender                     1259 non-null   object
 3   Country                    1259 non-null   object
 4   state                      744 non-null    object
 5   self_employed              1241 non-null   object
 6   family_history             1259 non-null   object
 7   treatment                  1259 non-null   object
 8   work_interfere             995 non-null    object
 9   no_employees               1259 non-null   object
 10  remote_work                1259 non-null   object
 11  tech_company               1259 non-null   object
 12  benefits                   1259 non-null   object
 13  care_options               1259 non-null   object
 14  wellness